# Task 15 — Reproduce concept training before extending it

This notebook answers one question first: **can we reproduce Zhang, Jurafsky, and Shani on the released C4 setup?** It keeps the released gold-inclusive set-marginal objective unchanged. Our uniform and contrastive losses are not run here.

| Readable method | What changes |
|---|---|
| Pretrained | No post-training |
| NTP fine-tuning | Ordinary next-token training on the same C4 rows |
| Augmented NTP | Synonym substitutions are written into training text |
| Randomized concepts | Same-sized incoherent sets; semantic control |
| Zhang concept marginal | Raises total probability of the observed word plus accepted alternatives |

Primary reproduction evidence is nine-task mean STS, content-word NTP, and global NTP. The untouched model need not lose to NTP on every STS task.


In [ ]:
from pathlib import Path
from getpass import getpass
import hashlib, json, os, shutil, subprocess, sys, torch

BASE_MODEL = "meta-llama/Llama-3.2-1B"
PRIMARY_SEED = 42
# One seed first.  Seed 42 alone screens the pipeline and shows the direction of
# every effect, but it CANNOT support a claim: the pre-registered rule needs all
# three seeds to agree in sign.  Flip to True for the reportable run; resume makes
# the seed-42 arms free the second time.
RUN_MULTISEED = False
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"
MAIN = Path("/content/concept-aware-training")
EXT = Path("/content/learning-concepts")
DATA = Path("/content/concept_data")
RUNS = Path("/content/concept_runs")
DRIVE_PROJECT = Path("/content/drive/MyDrive/concept_training")
DRIVE_RESULTS = DRIVE_PROJECT / "task15_16_results"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache/datasets"
os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache/transformers"

# Turn on one gate at a time.  Defaults are safe and do not start a GPU matrix.
RUN_DATA = False
RUN_SMOKE = False
RUN_SCREEN = False
RUN_CONFIRM = False
RUN_EVAL = False
# Skip any run whose adapter is already on Drive.  /content is wiped between
# Colab sessions, so without this the screen -> gate -> confirm sequence has to
# finish in one sitting.  Set False to force a full retrain.
RESUME_FINISHED_RUNS = True

def run(argv, cwd=None, env=None):
    print("+", " ".join(map(str, argv)))
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(DRIVE_RESULTS)})
    if env: merged.update(env)
    subprocess.run(list(map(str, argv)), cwd=cwd, env=merged, check=True)

def assert_ephemeral(path):
    resolved = str(Path(path).resolve())
    assert resolved.startswith("/content/") and not resolved.startswith("/content/drive/"), resolved

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, label):
    destination = DRIVE_RESULTS / label
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_no_drive_weights():
    # LoRA adapters at r=4 are ~10 MB and ARE cached to Drive on purpose: /content is
    # ephemeral, and without them a disconnect between the screen and confirm phases
    # discards every trained arm.  Full weights and optimizer state stay off Drive.
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in DRIVE_RESULTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached Drive: {found}"

DATA_CACHE = DRIVE_PROJECT / "task15_16_data"

def cache_dataset_to_drive(leaf):
    """Cache JSONL data needed by later notebooks, including top-k shards."""
    model_root = Path(leaf).parent
    copied = 0
    for source in (model_root, model_root / "embedding", model_root / "prompting"):
        destination = DATA_CACHE / source.relative_to(DATA)
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            shutil.copy2(path, destination / path.name)
            copied += 1
    print(f"cached {copied} dataset files under", DATA_CACHE / model_root.relative_to(DATA))

def restore_dataset_from_drive(leaf):
    model_root = Path(leaf).parent
    embedding_source = DATA_CACHE / Path(leaf).relative_to(DATA)
    copied = 0
    for destination in (model_root, model_root / "embedding", model_root / "prompting"):
        source = DATA_CACHE / destination.relative_to(DATA)
        if not source.exists():
            continue
        destination.mkdir(parents=True, exist_ok=True)
        for path in source.glob("*.jsonl"):
            target = destination / path.name
            if not target.is_file():
                shutil.copy2(path, target)
                copied += 1
    print(f"restored {copied} dataset files from", DATA_CACHE / model_root.relative_to(DATA))
    return (embedding_source / "synonyms_train.jsonl").is_file()

RUN_MANIFEST = DRIVE_RESULTS / "run_manifests"

def save_runs(runs, name):
    """Persist label -> adapter path so a later session can evaluate earlier phases."""
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    if not path.is_file():
        return {}
    return {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    """Pull every adapter in `runs` back onto /content; drop any that is missing."""
    live = {}
    for label, path in runs.items():
        if (Path(path) / "adapter_config.json").is_file() or restore_adapter_from_drive(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    """True when a completed evaluation artifact is already on Drive."""
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

ADAPTER_CACHE = DRIVE_PROJECT / "task15_16_adapters"
ADAPTER_FILES = ("adapter_config.json", "adapter_model.safetensors",
                 "training_history.jsonl", "run_config.json")

def cache_adapter_to_drive(path):
    """Copy one finished adapter to Drive so a later session can resume."""
    destination = ADAPTER_CACHE / Path(path).relative_to(RUNS)
    destination.mkdir(parents=True, exist_ok=True)
    for name in ADAPTER_FILES:
        source = Path(path) / name
        if source.is_file():
            shutil.copy2(source, destination / name)
    return destination

def restore_adapter_from_drive(path):
    """Return True when a completed adapter for `path` was restored from Drive."""
    source = ADAPTER_CACHE / Path(path).relative_to(RUNS)
    if not (source / "adapter_config.json").is_file():
        return False
    Path(path).mkdir(parents=True, exist_ok=True)
    for name in ADAPTER_FILES:
        candidate = source / name
        if candidate.is_file():
            shutil.copy2(candidate, Path(path) / name)
    return True

DATA.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
run(["git", "checkout", "--detach", UPSTREAM_COMMIT], cwd=EXT)
patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), "Commit external/learning-concepts.patch before running Colab."
check = subprocess.run(["git", "apply", "--check", str(patch_file)], cwd=EXT)
if check.returncode == 0:
    run(["git", "apply", str(patch_file)], cwd=EXT)
else:
    reverse = subprocess.run(["git", "apply", "--reverse", "--check", str(patch_file)], cwd=EXT)
    assert reverse.returncode == 0, "External checkout is neither clean nor exactly patched."

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate",
     "peft", "bitsandbytes", "datasets", "spacy", "mteb>=1.12", "wandb",
     "nltk", "scipy", "scikit-learn", "seaborn", "pytest"])
run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", DRIVE_RESULTS / "external_benchmark_integrity.json"], cwd=MAIN)
(DRIVE_RESULTS / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)

from huggingface_hub import login
login(token=getpass("Hugging Face token (input hidden): "), add_to_git_credential=False)


## Rebuild and audit the exact 8k/1k/1k data

The audit hard-fails on split overlap, target misalignment, empty sets, or any concept that is not a complete single token. The observed target is part of every set by construction.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
if RUN_DATA:
    # The extraction is the longest stage.  Ten independent 1k shards are
    # cached after completion, so a Colab disconnect loses at most one shard.
    restore_dataset_from_drive(LEAF)
    combined = LEAF.parent / "combined.jsonl"
    if not combined.is_file():
        run([sys.executable, "data/get_content_words.py", "--model", BASE_MODEL,
             "--dataset", "c4", "--max_length", "256"], cwd=EXT)
        cache_dataset_to_drive(LEAF)
    for start in range(0, 10000, 1000):
        end = start + 1000
        synonym_part = LEAF / f"synonyms_{start}_{end}.jsonl"
        topk_part = LEAF.parent / "prompting" / f"topk_{start}_{end}.jsonl"
        if synonym_part.is_file() and topk_part.is_file():
            print("resume: extraction shard already complete", start, end)
            continue
        run([sys.executable, "data/embedding_synonyms.py", "c4",
             "--start", start, "--end", end, "--model", BASE_MODEL], cwd=EXT)
        cache_dataset_to_drive(LEAF)
    run([sys.executable, "data/merge_synonym_parts.py", "--train-size", "8000",
         "--val-size", "1000", "--test-size", "1000", "--expected-count", "2", "--force"], cwd=EXT)
    run([sys.executable, "data/augment_synonyms.py", "--base-dir", DATA,
         "--num-augmentations", "4", "--seed", "42", "--overwrite"], cwd=EXT)
    run([sys.executable, "data/randomize_synonyms.py", "--split", "train", "--overwrite"], cwd=EXT)

if RUN_DATA:
    cache_dataset_to_drive(LEAF)
if RUN_DATA:
    run([sys.executable, "data/audit_concept_data.py",
         "--train", LEAF / "synonyms_train.jsonl",
         "--validation", LEAF / "synonyms_val.jsonl",
         "--test", LEAF / "synonyms_test.jsonl",
         "--tokenizer", BASE_MODEL,
         "--report", DRIVE_RESULTS / "data_audit.json"], cwd=EXT)


## Unit tests and eight-row GPU smoke run

The smoke run checks the complete QLoRA path before any sweep. It is deleted immediately. The tests cover the released loss, our optional objectives, gradients, and hierarchy sequence scoring.


In [ ]:
if RUN_SMOKE:
    run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=EXT)
    # No map-style preprocessing cache is used. Two independent loads must
    # still produce identical candidate supervision.
    from train import ConceptDataset
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    first = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    second = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    assert [x["content_words"] for x in first.data] == [x["content_words"] for x in second.data]
    smoke = RUNS / "smoke"
    assert_ephemeral(smoke)
    run([sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
         "--dataset-type", "embedding", "--concept-loss-weight", "1.0",
         "--max-train-samples", "8", "--num-train-epochs", "1",
         "--output-dir", smoke, "--save-strategy", "no", "--report-to", "none"], cwd=EXT)
    assert (smoke / "adapter_config.json").exists()
    # Required one-model equivalence test: adapter logits and merged logits.
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    adapted = PeftModel.from_pretrained(base, smoke).eval()
    probe = tokenizer("A dog is an animal.", return_tensors="pt")
    with torch.no_grad(): adapter_logits = adapted(**probe).logits
    merged = adapted.merge_and_unload().eval()
    with torch.no_grad(): merged_logits = merged(**probe).logits
    assert torch.allclose(adapter_logits, merged_logits, atol=2e-4, rtol=2e-4)
    del merged, adapted, base
    shutil.rmtree(smoke)


## Exact reproduction schedule

Seed 42 gets the complete $\lambda\in\{0.25,0.5,0.75,1\}$ curve. The headline NTP, one-epoch augmented NTP, randomized $\lambda=.25$, and concept-marginal $\lambda=1$ settings are then confirmed with seeds 42, 123, and 2024. Released effective batch size is logged. If the headline fails, only seed 42 is rerun with the paper-stated batch before any diagnosis.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# /content is wiped between sessions; pull the generated concept data back rather
# than paying the multi-hour regeneration again.
restore_dataset_from_drive(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / method / f"seed_{seed}" / str(value)
    assert_ephemeral(path)
    return path

def finished(path):
    """A run counts as finished when its adapter exists locally or on Drive."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter_from_drive(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2):
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter_to_drive(out)
    return out


In [ ]:
REPRO_RUNS = load_runs("task15")
if RUN_SCREEN:
    REPRO_RUNS["ntp_seed42"] = train_flat("ntp", 42, 0.0)
    for lam in [0.25, 0.5, 0.75, 1.0]:
        REPRO_RUNS[f"zhang_lambda{lam}_seed42"] = train_flat("zhang_marginal", 42, lam)
    REPRO_RUNS["augmented_ntp_seed42"] = train_flat("augmented_ntp", 42, 0.0, epochs=1,
        train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
    REPRO_RUNS["randomized_seed42"] = train_flat("randomized", 42, 0.25, randomized=True)

if RUN_CONFIRM:
    for seed in SEEDS:
        REPRO_RUNS[f"ntp_seed{seed}"] = train_flat("ntp", seed, 0.0)
        REPRO_RUNS[f"augmented_ntp_seed{seed}"] = train_flat("augmented_ntp", seed, 0.0,
            epochs=1, train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
        REPRO_RUNS[f"randomized_seed{seed}"] = train_flat("randomized", seed, 0.25, randomized=True)
        REPRO_RUNS[f"zhang_seed{seed}"] = train_flat("zhang_marginal", seed, 1.0)
    # The lambda sweep already trained zhang_marginal at lambda=1.0 on seed 42, and
    # adapter_path() maps both calls to the same directory.  Two dict keys pointing at
    # one adapter would score it twice and report it as two arms.
    REPRO_RUNS.pop("zhang_lambda1.0_seed42", None)
save_runs(REPRO_RUNS, "task15")

# Use only if the released-batch seed-42 reproduction misses the stated trend.
# This is a named sensitivity run, never a replacement or a cherry-picked row.
RERUN_PAPER_STATED_BATCH = False
if RERUN_PAPER_STATED_BATCH:
    REPRO_RUNS["zhang_seed42_paper_batch"] = train_flat(
        "zhang_marginal_paper_batch", 42, 1.0, batch=2, accum=1)


## Evaluation and learning curves

Every checkpoint is scored by the same evaluators. “Global NLL” covers every next-token position; “content-word NLL” covers the semantic slots; “set mass” is total probability assigned to the gold-inclusive valid set. SWORDS and bm-semlex are zero-shot here.


In [ ]:
if RUN_EVAL:
    # A fresh session has the manifest but not the weights; pull them back first.
    REPRO_RUNS = restore_all(REPRO_RUNS)
    checkpoints = [BASE_MODEL, *map(str, REPRO_RUNS.values())]
    result_dir = DRIVE_RESULTS / "task15_reproduction"
    result_dir.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
         "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
         "--output", result_dir / "perplexity.json"], cwd=EXT)
    run([sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
         "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
         "--output", result_dir / "concept_sets.json"], cwd=EXT)
    for label, checkpoint in {"pretrained": BASE_MODEL, **REPRO_RUNS}.items():
        display_label = label.replace("_", " ")
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label,
                     "--csv-output", result_dir / f"sts_{display_label}.csv",
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
         "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
         "--results_json", result_dir / "swords_zero_shot.json", "--modes", "left", "full"], cwd=MAIN)
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
         "--kind", "swords", "--results-json", result_dir / "swords_zero_shot.json",
         "--baseline-index", "0", "--output", result_dir / "swords_paired_ci_vs_pretrained.json"], cwd=MAIN)
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
         "--data", MAIN / "data/bm_semlex/curated_200.tsv",
         "--results_json", result_dir / "bm_semlex.json"], cwd=MAIN)
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in REPRO_RUNS.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_main_table.csv"], cwd=MAIN)
    for label, path in REPRO_RUNS.items(): sync_small_artifacts(path, f"task15_logs/{label}")
    audit_no_drive_weights()


In [ ]:
# Plot only scalar learning curves; adapters remain under /content.
import pandas as pd, seaborn as sns
from matplotlib import pyplot as plt
curves = []
for label, path in REPRO_RUNS.items():
    history = Path(path) / "training_history.jsonl"
    if history.exists():
        frame = pd.read_json(history, lines=True)
        frame["method"] = label
        curves.append(frame)
if curves:
    frame = pd.concat(curves, ignore_index=True)
    value = "loss" if "loss" in frame else "ce_loss"
    sns.lineplot(frame, x="step", y=value, hue="method")
    plt.tight_layout(); plt.savefig(DRIVE_RESULTS / "task15_reproduction" / "training_loss.png", dpi=180)


## Reproduction gate

Proceed only if Zhang concept marginal beats NTP and augmented NTP on mean STS, improves content-word perplexity over NTP, stays near pretrained global perplexity, and the $\lambda$ curve has the reported direction. If seed 42 fails, rerun that one setting with the paper-stated effective batch and report both configurations—do not silently substitute it.
